# FedPSPO: Federated Proximal Smoothing Policy Optimization

Code for the paper *FedPSPO*. A PPO agent with temporal action smoothing sets the client learning rate, local epochs, server aggregation weight, and the numbers of selected features and client clusters in every federated round.

**Setup used in the paper:** ResNet-18 at 224×224, 4 clients, Dirichlet α = 0.5 label skew, 40 rounds, seeds 42, 7 and 123. Datasets: BloodMNIST, DermaMNIST, PneumoniaMNIST (MedMNIST v2). Baselines: FedAvg, FedProx, SCAFFOLD.

## How to run (Kaggle, GPU)
1. Attach the MedMNIST v2 `.npz` files as inputs: `bloodmnist_224.npz`, `dermamnist_224.npz`, `pneumoniamnist_224.npz` (available from https://medmnist.com/ / Zenodo). The 224-px versions are used when present.
2. Run the cells in order. Results go to `/kaggle/working/fedpspo_outputs/` as one JSON per run.
3. Runs that are already finished are skipped. If the session time limit is reached, save the version, attach its output as an input, and run again to continue.

| Cell | What it does | Paper |
|---|---|---|
| 1 | All seed-42 runs: 4 methods × 3 datasets, 7 ablations, Dirichlet 0.1 / 1.0 sweep; prints `SUMMARY.txt` and figures | Tables of results, ablation, non-IID sweep |
| 2 | Seeds 7 and 123 for the 12 main runs; prints mean ± std over 3 seeds (`SUMMARY_SEEDS.txt`) | Main results (mean ± std), win counts |
| 3 | Test-accuracy curves per round, mean over seeds (`curves_all.pdf`) | Training-curve figure |

## 1. Main experiments (seed 42)

In [ ]:
# ===================== FedPSPO: full experiment cell (Kaggle) =====================
import os, re, glob, json, time, math, copy, random, shutil
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, f1_score, recall_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.rcParams.update({"font.size":15,"axes.titlesize":17,"axes.labelsize":16,"legend.fontsize":13})

OUT="/kaggle/working/fedpspo_outputs"; os.makedirs(OUT,exist_ok=True)
START=time.time(); TIME_BUDGET_H=11.3
SEED=42; NUM_CLIENTS=4; ROUNDS=40; BATCH=64; WORKERS=2; IMG=224
INIT=dict(lr=0.01,E=5,lam=0.1,N=256,K=2)                      # paper's initial hyperparameters
BOUNDS=dict(lr=(1e-4,1e-1),E=(1,5),lam=(0.05,1.5),N=(32,512))  # Eq.11 constraints (report these)
PPO=dict(clip=0.2,epochs=4,buf=4,gamma=0.99,gae=0.95,lr=3e-4)
SMOOTH=0.25; BETA=(1.0,1.0,0.5); PROX_MU=0.01; PROBE=256
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP=DEV.type=="cuda"
print("Device:",torch.cuda.get_device_name(0) if AMP else "CPU")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---------------- data ----------------
def find_npz(name):
    c=[p for p in glob.glob("/kaggle/input/**/*.npz",recursive=True)
       if re.fullmatch(fr"{name}(_\d+)?\.npz",os.path.basename(p))]
    if not c: return None
    size=lambda p:int(re.search(r"_(\d+)\.npz$",p).group(1)) if re.search(r"_(\d+)\.npz$",p) else 28
    return max(c,key=size)

def load(name):
    d=np.load(find_npz(name))
    g=lambda s:(d[f"{s}_images"],d[f"{s}_labels"].reshape(-1).astype(np.int64))
    return g("train"),g("val"),g("test")

class MedDS(Dataset):
    def __init__(s,X,y,train):
        s.X,s.y=X,y; ops=[transforms.ToPILImage()]
        if X.shape[1]>=200: ops.append(transforms.CenterCrop(200))   # paper: center-crop 200 -> resize 224
        ops.append(transforms.Resize((IMG,IMG)))
        if train: ops.append(transforms.RandomHorizontalFlip())
        ops.append(transforms.ToTensor()); s.tf=transforms.Compose(ops)
        s.norm=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
    def __len__(s): return len(s.y)
    def raw(s,i):
        x=s.X[i]
        if x.ndim==3 and x.shape[-1]==1: x=x[...,0]
        t=s.tf(x.astype(np.uint8))
        return t.repeat(3,1,1) if t.shape[0]==1 else t
    def __getitem__(s,i): return s.norm(s.raw(i)), s.y[i]

def dirichlet_split(y,a,seed):
    rng=np.random.default_rng(seed)
    while True:
        idx=[[] for _ in range(NUM_CLIENTS)]
        for c in np.unique(y):
            ids=rng.permutation(np.where(y==c)[0])
            cuts=(np.cumsum(rng.dirichlet([a]*NUM_CLIENTS))*len(ids)).astype(int)[:-1]
            for k,p in enumerate(np.split(ids,cuts)): idx[k]+=p.tolist()
        if min(map(len,idx))>=2*BATCH: return [np.array(i) for i in idx]

_CACHE={}
def get_data(ds,a):
    if (ds,a) in _CACHE: return _CACHE[(ds,a)]
    (Xt,yt),(Xv,yv),(Xs,ys)=load(ds); nc=int(max(yt.max(),yv.max(),ys.max())+1)
    tr,va,te=MedDS(Xt,yt,True),MedDS(Xv,yv,False),MedDS(Xs,ys,False)
    parts=dirichlet_split(yt,a,SEED)
    mk=lambda d,s:DataLoader(d,BATCH,shuffle=s,num_workers=WORKERS,pin_memory=True)
    c0=np.bincount(ys).argmax()
    D=dict(cls=[mk(Subset(tr,p),True) for p in parts],sizes=[len(p) for p in parts],
           vl=mk(va,False),tl=mk(te,False),nc=nc,te=te,lat_idx=np.where(ys==c0)[0][:2].tolist(),
           counts=[np.bincount(yt[p],minlength=nc).tolist() for p in parts])
    _CACHE.clear(); _CACHE[(ds,a)]=D; return D

# ---------------- model / training ----------------
def build(nc):
    m=models.resnet18(weights=None); m.fc=nn.Linear(512,nc)
    return m.to(DEV).to(memory_format=torch.channels_last)

def flat(m): return torch.cat([p.detach().flatten() for p in m.parameters()])

@torch.no_grad()
def evaluate(m,loader):
    m.eval(); L=n=0; P=[]; Y=[]
    for x,y in loader:
        x,y=x.to(DEV,non_blocking=True),y.to(DEV)
        with torch.autocast("cuda",enabled=AMP): o=m(x).float()
        L+=F.cross_entropy(o,y,reduction="sum").item(); n+=len(y); P.append(o.softmax(1).cpu()); Y.append(y.cpu())
    P=torch.cat(P).numpy(); Y=torch.cat(Y).numpy(); pr=P.argmax(1)
    auc=roc_auc_score(Y,P[:,1]) if P.shape[1]==2 else roc_auc_score(Y,P,multi_class="ovr")
    return dict(loss=L/n,acc=float((pr==Y).mean()),auc=float(auc),pred=pr,y=Y,
                f1=float(f1_score(Y,pr,average="macro")),recall=float(recall_score(Y,pr,average="macro")))

def local_train(m,loader,lr,E,mode="plain",g=None,cg=None,cl=None):
    opt=torch.optim.SGD(m.parameters(),lr=lr,momentum=0 if mode=="scaffold" else 0.9,weight_decay=1e-4)
    sc=torch.cuda.amp.GradScaler(enabled=AMP); m.train(); steps=0
    for _ in range(E):
        for x,y in loader:
            x,y=x.to(DEV,non_blocking=True),y.to(DEV)
            with torch.autocast("cuda",enabled=AMP):
                loss=F.cross_entropy(m(x),y)
            if mode=="fedprox":
                loss=loss+PROX_MU/2*sum(((p-q)**2).sum() for p,q in zip(m.parameters(),g))
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward()
            if mode=="scaffold":
                sc.unscale_(opt)
                for p,a,b in zip(m.parameters(),cg,cl): p.grad+=a-b
            sc.step(opt); sc.update(); steps+=1
    return steps

@torch.no_grad()
def client_embedding(m,loader):
    m.eval(); body=nn.Sequential(*list(m.children())[:-1]); f=[]; seen=0
    for x,_ in loader:
        f.append(body(x.to(DEV)).flatten(1).float()); seen+=len(x)
        if seen>=PROBE: break
    return torch.cat(f).mean(0)

def aggregate(gm,locs,sizes,lam,embs=None,N=512,K=1):
    """W_{t+1}=W_t+lam*sum_c (1/|C|) sum_{k in c} (n_k/n_c) dW_k ; K=1 -> FedAvg weighting n_k/n."""
    W=gm.state_dict(); keys=[k for k in W if W[k].is_floating_point()]
    deltas=[]
    for l in locs:
        sd=l.state_dict(); deltas.append({k:sd[k]-W[k] for k in keys})
    lab=np.zeros(len(locs),int)
    if embs is not None and K>1:
        E=torch.stack(embs); top=E.var(0).topk(min(N,E.shape[1])).indices
        lab=KMeans(min(K,len(locs)),n_init=10,random_state=SEED).fit_predict(E[:,top].cpu().numpy())
    U=np.unique(lab); agg={k:torch.zeros_like(W[k]) for k in keys}
    for c in U:
        mem=np.where(lab==c)[0]; nc=sum(sizes[i] for i in mem)
        wc=(1/len(U)) if len(U)>1 else 1.0
        for k in keys: agg[k]+=wc*sum((sizes[i]/nc)*deltas[i][k] for i in mem)
    for k in keys: W[k]=W[k]+lam*agg[k]
    gm.load_state_dict(W); return lab.tolist()

# ---------------- PPO agent ----------------
class ActorCritic(nn.Module):
    def __init__(s,h=64):
        super().__init__()
        s.pi=nn.Sequential(nn.Linear(5,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,5))
        s.v=nn.Sequential(nn.Linear(5,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,1))
        s.log_std=nn.Parameter(torch.full((5,),-0.5))
    def dist(s,x): return torch.distributions.Normal(s.pi(x),s.log_std.exp())

def to_h(a):
    s=torch.sigmoid(a).tolist(); B=BOUNDS; lo,hi=map(math.log10,B["lr"])
    return dict(lr=10**(lo+s[0]*(hi-lo)),E=B["E"][0]+s[1]*(B["E"][1]-B["E"][0]),
                lam=B["lam"][0]+s[2]*(B["lam"][1]-B["lam"][0]),N=B["N"][0]+s[3]*(B["N"][1]-B["N"][0]),
                K=1+s[4]*(NUM_CLIENTS-1))

def smooth(prev,new,al):                       # Eq.3: a~_t = al*a_{t-1} + (1-al)*a_t
    out={k:al*prev[k]+(1-al)*new[k] for k in new}
    for k in("E","N","K"): out[k]=max(1,int(round(out[k])))
    return out

def ppo_update(ac,opt,buf,last_v):
    S=torch.stack([b[0] for b in buf]); A=torch.stack([b[1] for b in buf]); LP=torch.stack([b[2] for b in buf])
    R=[b[3] for b in buf]; V=[b[4] for b in buf]+[last_v]; adv=[]; g=0.
    for t in reversed(range(len(buf))):
        d=R[t]+PPO["gamma"]*V[t+1]-V[t]; g=d+PPO["gamma"]*PPO["gae"]*g; adv.insert(0,g)
    adv=torch.tensor(adv,device=DEV,dtype=torch.float32); ret=adv+torch.tensor(V[:-1],device=DEV,dtype=torch.float32)
    adv=(adv-adv.mean())/(adv.std()+1e-8); ls=[]
    for _ in range(PPO["epochs"]):
        d=ac.dist(S); r=(d.log_prob(A).sum(-1)-LP).exp()
        pl=-torch.min(r*adv,r.clamp(1-PPO["clip"],1+PPO["clip"])*adv).mean()
        loss=pl+0.5*F.mse_loss(ac.v(S).squeeze(-1),ret)-0.01*d.entropy().sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); ls.append(loss.item())
    return float(np.mean(ls))

@torch.no_grad()
def latent_maps(m,D):
    m.eval(); body=nn.Sequential(*list(m.children())[:-2]); out=[]
    for i in D["lat_idx"]:
        x,_=D["te"][i]; out.append(body(x[None].to(DEV)).float().mean(1)[0].cpu().numpy().tolist())
    return out

# ---------------- one federated run ----------------
def run(c):
    path=f"{OUT}/{c['name']}.json"
    set_seed(SEED); D=get_data(c["ds"],c["dir"]); M=c["method"]; ps=M=="fedpspo"
    gm=build(D["nc"]); h={**INIT,**c["force"]}; log=[]; snaps={}; rl=float("nan")
    if ps: ac=ActorCritic().to(DEV); aopt=torch.optim.Adam(ac.parameters(),PPO["lr"]); buf=[]
    if M=="scaffold":
        cg=[torch.zeros_like(p) for p in gm.parameters()]; cl=[[torch.zeros_like(p) for p in gm.parameters()] for _ in D["cls"]]
    ev=evaluate(gm,D["vl"]); L,A=ev["loss"],ev["acc"]
    state=torch.tensor([L,A,0.,0.,1.],device=DEV); Wp=flat(gm)
    for t in range(1,ROUNDS+1):
        t0=time.time()
        if ps:
            with torch.no_grad():
                d=ac.dist(state); a=d.sample(); lp=d.log_prob(a).sum(); v=ac.v(state).item()
            if t>1: h={**smooth(h,to_h(a),c["smooth"]),**c["force"]}
        lam=h["lam"] if ps else 1.0; E=int(h["E"])
        g=[p.detach().clone() for p in gm.parameters()]; locs=[]; embs=[]
        for i,ld in enumerate(D["cls"]):
            lm=copy.deepcopy(gm)
            if M=="scaffold":
                st=local_train(lm,ld,h["lr"],E,"scaffold",cg=cg,cl=cl[i])
                cl[i]=[ci-a_+(gj-p.detach())/(st*h["lr"]) for ci,a_,gj,p in zip(cl[i],cg,g,lm.parameters())]
            else:
                local_train(lm,ld,h["lr"],E,"fedprox" if M=="fedprox" else "plain",g=g)
            if ps: embs.append(client_embedding(lm,ld))
            locs.append(lm)
        if M=="scaffold": cg=[sum(x[j] for x in cl)/len(cl) for j in range(len(cg))]
        ta=time.time()
        clusters=aggregate(gm,locs,D["sizes"],lam,embs if ps else None,int(h["N"]),int(h["K"]) if ps else 1)
        del locs
        ev=evaluate(gm,D["vl"]); W=flat(gm); cos=F.cosine_similarity(W,Wp,dim=0).item(); Wp=W
        if ps:
            b1,b2,b3=c["beta"]; r=b1*ev["acc"]-b2*ev["loss"]+b3*cos            # Eq.8
            ns=torch.tensor([ev["loss"],ev["acc"],ev["loss"]-L,ev["acc"]-A,cos],device=DEV)
            buf.append((state,a,lp,r,v))
            if len(buf)>=PPO["buf"]:
                with torch.no_grad(): lv=ac.v(ns).item()
                rl=ppo_update(ac,aopt,buf,lv); buf=[]
            state=ns
        over=time.time()-ta; L,A=ev["loss"],ev["acc"]; te=evaluate(gm,D["tl"])
        if c["snap"] and t in(1,ROUNDS): snaps[str(t)]=latent_maps(gm,D)
        log.append(dict(round=t,lr=h["lr"],E=E,lam=lam,N=int(h["N"]),K=int(h["K"]),clusters=clusters,
                        val_loss=L,val_acc=A,test_acc=te["acc"],cos=cos,rl_loss=rl,
                        round_sec=time.time()-t0,agg_rl_sec=over))
        print(f"[{c['name']}] R{t:02d} lr={h['lr']:.4g} E={E} lam={lam:.3f} N={int(h['N'])} K={int(h['K'])} "
              f"ValL={L:.4f} ValA={A*100:.2f} TestA={te['acc']*100:.2f} ({time.time()-t0:.0f}s)",flush=True)
    te=evaluate(gm,D["tl"])
    imgs=[D["te"].raw(i).permute(1,2,0).numpy().tolist() for i in D["lat_idx"]] if c["snap"] else []
    res=dict(cfg=c,log=log,acc=te["acc"],auc=te["auc"],f1=te["f1"],recall=te["recall"],
             pred=te["pred"].tolist(),y=te["y"].tolist(),client_counts=D["counts"],snaps=snaps,lat_imgs=imgs)
    json.dump(res,open(path,"w")); torch.save(gm.state_dict(),f"{OUT}/{c['name']}.pt")
    return res

# ---------------- experiment plan (priority order) ----------------
RUNS=[]
def add(name,ds,method,dir=0.5,smooth=SMOOTH,beta=BETA,force=None,snap=False):
    RUNS.append(dict(name=name,ds=ds,method=method,dir=dir,smooth=smooth,beta=list(beta),force=force or {},snap=snap))
for m in("fedpspo","fedavg","fedprox","scaffold"): add(f"bloodmnist_{m}","bloodmnist",m,snap=(m=="fedpspo"))
ABL={"no_policy_smoothing":dict(smooth=0.0),"no_cosine_reward":dict(beta=(1,1,0)),
     "fixed_client_lr":dict(force={"lr":INIT["lr"]}),"fixed_local_epochs":dict(force={"E":INIT["E"]}),
     "fixed_server_weight":dict(force={"lam":INIT["lam"]}),"no_feature_selection":dict(force={"N":512}),
     "no_clustering":dict(force={"K":1})}
for k,v in ABL.items(): add(f"bloodmnist_abl_{k}","bloodmnist","fedpspo",**v)
for a in(0.1,1.0):
    for m in("fedpspo","fedavg"): add(f"bloodmnist_dir{a}_{m}","bloodmnist",m,dir=a)
for ds in("dermamnist","pneumoniamnist"):
    for m in("fedpspo","fedavg","fedprox","scaffold"): add(f"{ds}_{m}",ds,m)

# ---------------- report: tables + figures ----------------
def report():
    R={os.path.basename(p)[:-5]:json.load(open(p)) for p in glob.glob(f"{OUT}/*.json")}
    lines=[]
    def P(s=""): print(s); lines.append(s)
    meths=("fedavg","fedprox","scaffold","fedpspo")
    for ds in("bloodmnist","dermamnist","pneumoniamnist"):
        ms=[m for m in meths if f"{ds}_{m}" in R]
        if not ms: continue
        P(f"\n===== {ds}: {NUM_CLIENTS} clients, Dirichlet 0.5, ResNet-18, {ROUNDS} rounds =====")
        for m in ms:
            r=R[f"{ds}_{m}"]; lg=r["log"]
            P(f"{m:9s} acc={r['acc']*100:.2f}  auc={r['auc']*100:.2f}  macroF1={r['f1']:.4f}  macroRecall={r['recall']:.4f}  "
              f"sec/round={np.mean([l['round_sec'] for l in lg]):.1f}  agg+PPO sec/round={np.mean([l['agg_rl_sec'] for l in lg]):.2f}")
        if f"{ds}_fedpspo" in R:
            y=np.array(R[f"{ds}_fedpspo"]["y"]); p1=np.array(R[f"{ds}_fedpspo"]["pred"])
            for b in ms:
                if b=="fedpspo": continue
                p2=np.array(R[f"{ds}_{b}"]["pred"]); c1,c2=p1==y,p2==y
                tbl=[[int((c1&c2).sum()),int((c1&~c2).sum())],[int((~c1&c2).sum()),int((~c1&~c2).sum())]]
                mr=mcnemar(tbl,exact=False,correction=True)
                P(f"McNemar FedPSPO vs {b}: b={tbl[0][1]} c={tbl[1][0]} chi2={mr.statistic:.3f} p={mr.pvalue:.4g}")
            P(f"Per-client class counts: {R[f'{ds}_fedpspo']['client_counts']}")
        a,b=R.get(f"{ds}_fedavg"),R.get(f"{ds}_fedpspo")
        if a and b:
            r_=[l["round"] for l in b["log"]]
            fig,ax=plt.subplots(1,3,figsize=(21,6))
            ax[0].plot(r_,[l["rl_loss"] for l in b["log"]],"o-"); ax[0].set_title("(a) FedPSPO PPO loss"); ax[0].set_xlabel("Round")
            for n,x in(("FedAvg",a),("FedPSPO",b)):
                ax[1].plot(r_,[l["val_loss"] for l in x["log"]],lw=2,label=n)
                ax[2].plot(r_,[l["test_acc"]*100 for l in x["log"]],lw=2,label=f"{n} test")
                ax[2].plot(r_,[l["val_acc"]*100 for l in x["log"]],"--",label=f"{n} val")
            ax[1].set_title("(b) Validation loss"); ax[2].set_title("(c) Accuracy (%)")
            for k in(1,2): ax[k].legend(); ax[k].set_xlabel("Round")
            plt.tight_layout(); plt.savefig(f"{OUT}/{ds}_curves.pdf"); plt.close()
            fig,ax=plt.subplots(1,2,figsize=(18,8))
            for k,(n,x) in enumerate((("(a) FedAvg",a),("(b) FedPSPO",b))):
                cm=confusion_matrix(x["y"],x["pred"]); ax[k].imshow(cm,cmap="Blues"); ax[k].set_title(n)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]): ax[k].text(j,i,cm[i,j],ha="center",va="center",fontsize=12,
                                                             color="white" if cm[i,j]>cm.max()/2 else "black")
                ax[k].set_xlabel("Predicted"); ax[k].set_ylabel("True")
            plt.tight_layout(); plt.savefig(f"{OUT}/{ds}_confusion.pdf"); plt.close()
            cc=np.array(b["client_counts"]); fig,ax=plt.subplots(figsize=(10,6)); bot=np.zeros(len(cc))
            for j in range(cc.shape[1]): ax.bar(range(len(cc)),cc[:,j],bottom=bot,label=f"class {j}"); bot+=cc[:,j]
            ax.set_xticks(range(len(cc))); ax.set_xticklabels([f"Client {i+1}" for i in range(len(cc))])
            ax.set_ylabel("Samples"); ax.legend(ncol=2); plt.tight_layout(); plt.savefig(f"{OUT}/{ds}_client_split.pdf"); plt.close()
        if b and b["snaps"]:
            S=b["snaps"]; ks=sorted(S,key=int); n=len(b["lat_imgs"])
            fig,ax=plt.subplots(n,1+len(ks),figsize=(6*(1+len(ks)),5.5*n),squeeze=False)
            for r in range(n):
                ax[r,0].imshow(np.array(b["lat_imgs"][r])); ax[r,0].set_title("Input"); ax[r,0].axis("off")
                for j,k in enumerate(ks,1):
                    im=ax[r,j].imshow(np.array(S[k][r]),cmap="viridis"); ax[r,j].set_title(f"Round {k}")
                    ax[r,j].axis("off"); fig.colorbar(im,ax=ax[r,j],fraction=0.046)
            plt.tight_layout(); plt.savefig(f"{OUT}/{ds}_latent.pdf"); plt.close()
    ab=[(k,R[f"bloodmnist_abl_{k}"]["acc"]*100) for k in ABL if f"bloodmnist_abl_{k}" in R]
    if ab and "bloodmnist_fedpspo" in R:
        P("\n===== Ablation (BloodMNIST, test acc %) =====")
        rows=[("FedPSPO (full)",R["bloodmnist_fedpspo"]["acc"]*100)]+ab
        if "bloodmnist_fedavg" in R: rows.append(("FedAvg",R["bloodmnist_fedavg"]["acc"]*100))
        for n,v in rows: P(f"{n:25s} {v:.2f}")
        fig,ax=plt.subplots(figsize=(12,7)); ax.barh([r[0] for r in rows][::-1],[r[1] for r in rows][::-1])
        for i,(n,v) in enumerate(rows[::-1]): ax.text(v+0.1,i,f"{v:.2f}",va="center")
        ax.set_xlabel("Test accuracy (%)"); ax.set_xlim(min(r[1] for r in rows)-3,100)
        plt.tight_layout(); plt.savefig(f"{OUT}/bloodmnist_ablation.pdf"); plt.close()
    sw=[(a_,m) for a_ in(0.1,1.0) for m in("fedavg","fedpspo") if f"bloodmnist_dir{a_}_{m}" in R]
    if sw:
        P("\n===== Non-IID sweep (BloodMNIST, test acc %) =====")
        for a_,m in sw: P(f"Dirichlet {a_}  {m:8s} {R[f'bloodmnist_dir{a_}_{m}']['acc']*100:.2f}")
    if "bloodmnist_fedpspo" in R:
        lg=R["bloodmnist_fedpspo"]["log"]
        P(f"\nAction ranges actually used by FedPSPO: lr {min(l['lr'] for l in lg):.4g}-{max(l['lr'] for l in lg):.4g}, "
          f"E {min(l['E'] for l in lg)}-{max(l['E'] for l in lg)}, lam {min(l['lam'] for l in lg):.3f}-{max(l['lam'] for l in lg):.3f}, "
          f"N {min(l['N'] for l in lg)}-{max(l['N'] for l in lg)}, K {min(l['K'] for l in lg)}-{max(l['K'] for l in lg)}")
    missing=[c["name"] for c in RUNS if c["name"] not in R]
    P(f"\nCompleted {len(RUNS)-len(missing)}/{len(RUNS)} runs. Remaining: {missing}")
    open(f"{OUT}/SUMMARY.txt","w").write("\n".join(lines))

# ---------------- main: resume + run + report ----------------
for p in glob.glob("/kaggle/input/**/fedpspo_outputs/*.json",recursive=True):
    q=f"{OUT}/{os.path.basename(p)}"
    if not os.path.exists(q): shutil.copy(p,q)
durs=[]
for c in RUNS:
    if os.path.exists(f"{OUT}/{c['name']}.json"): print("already done:",c["name"]); continue
    if find_npz(c["ds"]) is None: print("dataset not attached, skipping:",c["ds"]); continue
    if time.time()-START+(np.mean(durs) if durs else 0)>TIME_BUDGET_H*3600:
        print("\n>>> Time budget reached. Save Version, add its output as input, and Run All again to continue."); break
    t0=time.time(); run(c); durs.append(time.time()-t0); report()
report()

## 2. Extra seeds (7 and 123) and mean ± std over three seeds
Uses `RUNS`, `run()` and the other globals from cell 1.

In [ ]:
# ===================== FedPSPO: extra seeds (paste as a NEW cell BELOW the main cell) =====================
# Reruns the 12 main configs (4 methods x 3 datasets) with seeds 7 and 123.
# Seed 42 results are reused as they are. Results are saved as <name>_s7.json / <name>_s123.json,
# so nothing is overwritten and finished runs are skipped after a restart.
# Prints mean +- std over the 3 seeds and writes SUMMARY_SEEDS.txt.

EXTRA_SEEDS = [7, 123]
DATASETS = ["dermamnist", "pneumoniamnist", "bloodmnist"]   # smallest gaps first, BloodMNIST last
METHODS = ["fedpspo", "fedavg", "fedprox", "scaffold"]
MAIN = {c["name"]: c for c in RUNS}                         # RUNS comes from the main cell

def seed_name(ds, m, s):
    return f"{ds}_{m}" if s == 42 else f"{ds}_{m}_s{s}"

# resume: pick up seed jsons saved in an earlier session (attach that version's output as input)
for p in glob.glob("/kaggle/input/**/fedpspo_outputs/*_s*.json", recursive=True):
    q = f"{OUT}/{os.path.basename(p)}"
    if not os.path.exists(q): shutil.copy(p, q)

durs = []
stop = False
for s in EXTRA_SEEDS:
    SEED = s            # run(), dirichlet_split() and KMeans read this global
    _CACHE.clear()      # the client split depends on the seed, so rebuild it
    for ds in DATASETS:
        for m in METHODS:
            c = dict(MAIN[f"{ds}_{m}"]); c["name"] = seed_name(ds, m, s); c["snap"] = False
            if os.path.exists(f"{OUT}/{c['name']}.json"): print("already done:", c["name"]); continue
            if find_npz(ds) is None: print("dataset not attached, skipping:", ds); continue
            if time.time() - START + (np.mean(durs) if durs else 0) > TIME_BUDGET_H * 3600:
                print("\n>>> Time budget reached. Save Version, add its output as input, and Run All again to continue.")
                stop = True; break
            t0 = time.time(); run(c); durs.append(time.time() - t0)
            p = f"{OUT}/{c['name']}.pt"
            if os.path.exists(p): os.remove(p)   # keep disk usage low; the .json has everything needed
        if stop: break
    if stop: break
SEED = 42

# ---------------- mean +- std report over seeds 42, 7, 123 ----------------
def seed_report():
    lines = []
    def P(x=""): print(x); lines.append(x)
    seeds = [42] + EXTRA_SEEDS
    for ds in ["bloodmnist", "dermamnist", "pneumoniamnist"]:
        P(f"\n===== {ds}: mean +- std over seeds {seeds} =====")
        accs = {}
        for m in ["fedavg", "fedprox", "scaffold", "fedpspo"]:
            rs = {s: f"{OUT}/{seed_name(ds, m, s)}.json" for s in seeds}
            rs = {s: json.load(open(p)) for s, p in rs.items() if os.path.exists(p)}
            if not rs: continue
            acc = np.array([r["acc"] * 100 for r in rs.values()]); auc = np.array([r["auc"] * 100 for r in rs.values()])
            f1 = np.array([r["f1"] for r in rs.values()]); rec = np.array([r["recall"] for r in rs.values()])
            sec = np.array([np.mean([l["round_sec"] for l in r["log"]]) for r in rs.values()])
            accs[m] = {s: r["acc"] * 100 for s, r in rs.items()}
            sd = lambda a: a.std(ddof=1) if len(a) > 1 else 0.0
            P(f"{m:9s} n={len(rs)}  acc={acc.mean():.2f}+-{sd(acc):.2f}  auc={auc.mean():.2f}+-{sd(auc):.2f}  "
              f"macroF1={f1.mean():.4f}+-{sd(f1):.4f}  macroRecall={rec.mean():.4f}+-{sd(rec):.4f}  sec/round={sec.mean():.1f}  "
              f"per-seed acc={[round(accs[m][s], 2) for s in sorted(accs[m])]}")
        if "fedpspo" in accs:
            for b in ["fedavg", "fedprox", "scaffold"]:
                if b not in accs: continue
                common = sorted(set(accs["fedpspo"]) & set(accs[b]))
                wins = sum(accs["fedpspo"][s] > accs[b][s] for s in common)
                diff = np.mean([accs["fedpspo"][s] - accs[b][s] for s in common])
                P(f"FedPSPO vs {b}: wins {wins}/{len(common)} seeds, mean acc difference {diff:+.2f}")
    open(f"{OUT}/SUMMARY_SEEDS.txt", "w").write("\n".join(lines))

seed_report()

## 3. Training-curve figure

In [ ]:
# ===================== FedPSPO: compact training-curve figure for the paper =====================
# Needs no GPU. Reads the 36 result JSONs from OUT (this session) or from an attached
# Kaggle input that contains fedpspo_outputs/ (e.g. a previous version of this notebook).
# Makes ONE full-width figure, curves_all.pdf: test accuracy per round on the three datasets,
# all four methods, mean over seeds 42/7/123 (no std band).
import glob, json, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = globals().get("OUT", "/kaggle/working/fedpspo_outputs"); os.makedirs(OUT, exist_ok=True)
SEEDS = [42, 7, 123]
DS = [("bloodmnist", "BloodMNIST"), ("dermamnist", "DermaMNIST"), ("pneumoniamnist", "PneumoniaMNIST")]
METHODS = [("fedavg", "FedAvg", "#1f77b4"), ("fedprox", "FedProx", "#2ca02c"),
           ("scaffold", "SCAFFOLD", "#9467bd"), ("fedpspo", "FedPSPO", "#d62728")]

def find(name):
    hits = [p for p in [f"{OUT}/{name}.json"] if os.path.exists(p)] or \
           glob.glob(f"/kaggle/input/**/fedpspo_outputs/{name}.json", recursive=True)
    return json.load(open(hits[0])) if hits else None

plt.rcParams.update({"font.size": 8, "axes.titlesize": 9, "legend.fontsize": 7,
                     "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7})
fig, axes = plt.subplots(1, 3, figsize=(7.16, 1.75))   # IEEE two-column text width
for ax, (ds, title) in zip(axes, DS):
    for m, label, col in METHODS:
        runs = [find(f"{ds}_{m}" if s == 42 else f"{ds}_{m}_s{s}") for s in SEEDS]
        runs = [r for r in runs if r]
        if not runs: print("missing:", ds, m); continue
        acc = np.array([[l["test_acc"] * 100 for l in r["log"]] for r in runs])
        x = [l["round"] for l in runs[0]["log"]]
        mu, sd = acc.mean(0), acc.std(0)
        ax.plot(x, mu, color=col, lw=1.6 if m == "fedpspo" else 1.0, label=label)
        # numbers for the paper text
        final = mu[-1]; r95 = next(i for i, v in zip(x, mu) if v >= 0.95 * final)
        print(f"{ds:15s} {m:9s} n={len(runs)} final={final:.2f}  first round >=95% of final: {r95}  "
              f"mean std over rounds={sd.mean():.2f}")
    ax.set_title(title); ax.set_xlabel("Round"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Test accuracy (%)")
axes[0].legend(loc="lower right", frameon=False)
plt.tight_layout(pad=0.3)
plt.savefig(f"{OUT}/curves_all.pdf", bbox_inches="tight")
plt.savefig(f"{OUT}/curves_all.png", dpi=200, bbox_inches="tight")
print("saved", f"{OUT}/curves_all.pdf")